In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats
from scipy.special import expit

In [ ]:
TASK = Path.cwd().parent
ROOT = TASK.parent
INPUT, OUTPUT = TASK / 'input', TASK / 'output'
CONFIG = json.loads((ROOT / 'config.json').read_text())
EPS, Y, SEED = CONFIG['eps'], CONFIG['last_year'], CONFIG['seed']

In [ ]:
N = CONFIG['n_sim']
rng = np.random.default_rng(SEED)

In [ ]:
TAG = 'lognormal_grw'
GENERATED_FROM = 1

In [ ]:
params = pd.read_csv(INPUT / TAG / 'parameters.csv').iloc[0]
panel = pd.read_csv(INPUT / 'data/panel.csv', index_col='dblp_id')
DEST = OUTPUT / TAG
DEST.mkdir(exist_ok=True)
sim = np.zeros((Y+1, N))

In [ ]:
# empirical initial distribution; iid shocks from the pooled normal fit
sim[0] = rng.choice(panel['0'].dropna().to_numpy(), N, replace=True)
z = np.log(sim[0]+EPS)
for t in range(Y):
    z_next = z + rng.normal(params.mu, params.sigma, N)
    sim[t+1] = np.maximum(np.exp(z_next)-EPS, 0)
    z = np.log(sim[t+1]+EPS)

In [ ]:
np.save(DEST / 'trajectories.npy', sim, allow_pickle=False)
pd.DataFrame([{'model':TAG, 'seed':SEED, 'n_sim':N, 'n_years':Y+1, 'generated_from_age':GENERATED_FROM,
               'orientation':'age x scholar', 'minimum':sim.min(), 'maximum':sim.max(),
               'finite':np.isfinite(sim).all()}]).to_csv(DEST / 'metadata.csv', index=False)
display(pd.read_csv(DEST / 'metadata.csv'))